In [ ]:
import datetime
import time
from pathlib import Path
from collections import Counter
import csv
from typing import List, Optional
from kiwipiepy import Kiwi

In [27]:
import datetime
import time
from pathlib import Path
from collections import Counter
import pandas as pd
from typing import List, Optional
from kiwipiepy import Kiwi

def analyze_word_frequency(
    target_words: Optional[List[str]] = None, 
    output_filename: str = "v99_빈도분석.xlsx",
    sheet_name: str = "빈도분석 결과",
    verbose: bool = True
) -> None:
    """
    1. 數 ~ 10. 望 폴더 내의 Markdown 파일들만 읽어 형태소 분석 및 빈도를 계산하고 Excel로 저장합니다.
    
    :param target_words: 특정 단어 리스트 (None 또는 빈 리스트일 경우 전체 단어 대상)
    :param output_filename: 저장할 Excel 파일명 (확장자 .xlsx 포함)
    :param sheet_name: Excel 내 저장할 시트명
    :param verbose: 실행 과정 및 실행시간 실시간 표시 여부 (기본값: True)
    """
    # ⏱️ 전체 시작 시간 측정
    total_start_time = time.time()
    
    ROOT = Path(r"/home/yhsong/gdrive/내 드라이브/my_vault/essay_v99")
    OUT = "/home/yhsong/gdrive/내 드라이브/my_vault/" + output_filename  # 지정된 파일명으로 경로 설정

    # 💡 MAG(일반부사)를 TARGET_TAGS에 추가
    TARGET_TAGS = {"NNG", "NNP", "VV", "VA", "MAG"}
    TAG_MAP = {
        "NNG": "일반명사",
        "NNP": "고유명사",
        "VV": "동사",
        "VA": "형용사",
        "MAG": "일반부사"
    }
    EXCLUDE_PATTERNS = ["집필 원칙", "목차", "단원별", "전체", "수식", "wordcount"]

    # 단어 리스트가 제공되었을 경우 set으로 변환하여 탐색 속도 최적화
    filter_words = set(target_words) if target_words else None

    # 💡 verbose 조건 수정 (verbose=True 일 때 출력)
    if verbose:
        current_time = datetime.datetime.now().strftime('%Y%m%d %H:%M:%S')
        print(f"[{current_time}] 🚀 1. 數 ~ 10. 望 폴더 대상 형태소 분석을 시작합니다...")

    # 안전장치: 원본 경로 확인
    if not ROOT.exists():
        raise FileNotFoundError(f"⚠️ 원본 폴더를 찾을 수 없습니다: {ROOT}")

    # 1부터 10까지의 폴더만 타겟 폴더 리스트로 지정
    target_folders = []
    for num in range(1, 11):
        matched_dirs = list(ROOT.glob(f"{num}.*"))
        for d in matched_dirs:
            if d.is_dir():
                target_folders.append(d)

    # 지정된 폴더들 내부에서만 .md 파일 수집
    file_list = []
    for folder in target_folders:
        for md in folder.rglob("*.md"):
            if not any(pat in md.name for pat in EXCLUDE_PATTERNS):
                file_list.append(md)
                
    total_files = len(file_list)

    kiwi = Kiwi()
    counter = Counter()  # (lemma, pos) -> count
    per_file = {}        # filename -> Counter

    # 1. 파일 분석 단계
    for idx, md in enumerate(file_list, 1):
        file_start_time = time.time()
        
        text = md.read_text(encoding="utf-8")
        file_counter = Counter()
        
        for token in kiwi.tokenize(text):
            if token.tag in TARGET_TAGS:
                if filter_words and token.form not in filter_words:
                    continue
                    
                key = (token.form, token.tag)
                counter[key] += 1
                file_counter[key] += 1
                
        per_file[md.relative_to(ROOT).as_posix()] = file_counter
        
        # 💡 verbose=True 일 때 실시간 진행 상황 및 파일당 소요 시간 출력
        if verbose:
            file_end_time = time.time()
            current_time = datetime.datetime.now().strftime('%Y%m%d %H:%M:%S')
            relative_path = md.relative_to(ROOT).as_posix()
            print(f"[{current_time}] 🔄 진행 중: ({idx}/{total_files}) {relative_path} 완료 | 소요 시간: {file_end_time - file_start_time:.4f}초")

    # 2. 데이터 정리 및 Excel 저장 단계
    save_start_time = time.time()
    
    # Counter 데이터를 데이터프레임으로 변환하기 위한 리스트 생성
    data = []
    for (form, tag), cnt in counter.most_common():
        korean_tag = TAG_MAP.get(tag, tag)
        data.append([form, korean_tag, cnt])
    
    df = pd.DataFrame(data, columns=["어휘", "품사", "전체 빈도"])
    
    # Excel 파일로 저장 (옵션으로 받은 sheet_name 사용)
    df.to_excel(OUT, index=False, sheet_name=sheet_name, engine="openpyxl")
            
    if verbose:
        save_end_time = time.time()
        current_time = datetime.datetime.now().strftime('%Y%m%d %H:%M:%S')
        print(f"[{current_time}] 💾 Excel 결과 저장 완료 ({sheet_name} 시트) | 소요 시간: {save_end_time - save_start_time:.4f}초")

    # 3. 최종 요약 출력
    total_end_time = time.time()
    current_time = datetime.datetime.now().strftime('%Y%m%d %H:%M:%S')
    
    print("-" * 60)
    print(f"[{current_time}] 🎉 최종 완료: {OUT}")
    print(f"📊 고유 형태소 수: {len(counter)}")
    print(f"📊 총 토큰 수: {sum(counter.values())}")
    print(f"⏱️ 총 실행 시간: {total_end_time - total_start_time:.2f}초")
    print("-" * 60)

In [28]:
words = ["세계", "우주", "자연", "현실", "실재"]
words = None
analyze_word_frequency(words, 
    output_filename="v99_핵심단어_분석.xlsx",
    sheet_name="부사_및_명사_추출",
    verbose=False)

------------------------------------------------------------
[20260524 21:19:55] 🎉 최종 완료: /home/yhsong/gdrive/내 드라이브/my_vault/v99_핵심단어_분석.xlsx
📊 고유 형태소 수: 3364
📊 총 토큰 수: 19307
⏱️ 총 실행 시간: 7.05초
------------------------------------------------------------
